In [ ]:
%%configure -f
{"vCores": 4, "defaultLakehouse": {"name": "diagnostic", "id": "9d10bce5-1edc-4875-83c4-ac0a98a02775", "workspaceId": "82ad2591-974a-4ad4-ace6-e24879274a4b"}}

# fabric-rlm 0.1.11.dev2 — **PVR OOD ablation** (RFP extraction + Spark log triage)

Two structured multi-field extraction tasks well outside the math/CS-hard training distribution.
Each runs **OFF** (`FABRIC_RLM_PVR=0`) and **ON** with **fresh bandit state**.


In [ ]:
import os, sys, json, time, traceback, uuid, platform as _platform, subprocess
from pathlib import Path
WHEEL_PATH = '/lakehouse/default/Files/fabric_rlm_longcot/wheels/fabric_rlm-0.1.11.dev2-py3-none-any.whl'
TIER = 'pvr_ood_ablation'
RUN_ID = time.strftime('%Y%m%d-%H%M%S') + '-' + uuid.uuid4().hex[:6]
FILES_ROOT = Path('/lakehouse/default/Files')
RUN_ROOT = FILES_ROOT / 'fabric_rlm_adaptive_validation' / TIER / RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=True)
SUMMARY_PATH = RUN_ROOT / 'summary.json'
summary = {'tier': TIER, 'run_id': RUN_ID, 'started_at': time.time(),
           'python': _platform.python_version(), 'wheel': WHEEL_PATH,
           'stages': [], 'ablation': {'cases': []}}
def write_summary():
    summary['elapsed_seconds'] = time.time() - summary['started_at']
    SUMMARY_PATH.write_text(json.dumps(summary, indent=2, default=str))
def stage(name, **info):
    summary['stages'].append({'stage': name, 't': round(time.time()-summary['started_at'],1), **info})
    write_summary(); print('[stage]', name, info)
stage('setup', run_root=str(RUN_ROOT))
subprocess.check_call(['pip','install','--quiet','--force-reinstall','--no-deps', WHEEL_PATH])
stage('pip_wheel', done=True)
subprocess.check_call(['pip','install','--quiet','dspy>=3.0.4'])
stage('pip_dspy', done=True)
import dspy, fabric_rlm
stage('imported', dspy=dspy.__version__, fabric_rlm=fabric_rlm.__version__)


In [ ]:
FIX = FILES_ROOT / 'fabric_rlm_adaptive_validation' / 'fixtures'
rfp_text = (FIX / 'rfp_sample_100k.txt').read_text(encoding='utf-8', errors='ignore')
spark_text = (FIX / 'spark_log_sample_200k.json').read_text(encoding='utf-8', errors='ignore')
stage('fixtures_loaded', rfp_chars=len(rfp_text), spark_chars=len(spark_text))

RFP_QUESTION = (
    'You are given the partial text of a 2025 Request for Proposal document. '
    'Extract the following four fields and return ONLY a single JSON object '
    'with exactly these keys: date_of_issue, response_due, authority, letter_of_credit_percent. '
    'date_of_issue and response_due must be ISO format YYYY-MM-DD. '
    'authority is the issuing organization name. '
    'letter_of_credit_percent is the integer percent of the Fixed Management Fee required as a Letter of Credit.\n\n'
    '--- RFP TEXT ---\n' + rfp_text
)
SPARK_QUESTION = (
    'You are given the first 200KB of a Spark event log (newline-delimited JSON events). '
    'Extract the following five fields and return ONLY a single JSON object with exactly these keys: '
    'spark_version, app_id, app_name, n_executors_added, gluten_version. '
    'n_executors_added is the integer count of SparkListenerExecutorAdded events found in the log slice.\n\n'
    '--- SPARK LOG ---\n' + spark_text
)

CASES = [
    {'id':'rfp-extract', 'template':'rfp_extract', 'difficulty':'ood',
     'question': RFP_QUESTION,
     'truth': {'date_of_issue':'2025-02-13','response_due':'2025-03-31',
               'authority_must_contain':'victoria',
               'letter_of_credit_percent':5}},
    {'id':'spark-extract','template':'spark_extract','difficulty':'ood',
     'question': SPARK_QUESTION,
     'truth': {'spark_version':'3.5.5.5.4.20251218.3',
               'app_id':'application_1769950908745_0001',
               'app_name_must_contain':'EHS_DATA_MASTER_NOTEBOOK',
               'n_executors_added':1,
               'gluten_version':'1.3.0-20251127.2'}},
]
stage('cases_built', n=len(CASES))


In [ ]:
import re
def parse_json_answer(s):
    if not s: return None
    s = s.strip()
    m = re.search(r'\{[\s\S]*\}', s)
    if not m: return None
    try: return json.loads(m.group(0))
    except Exception:
        try: return json.loads(m.group(0).replace("'",'"'))
        except Exception: return None

def make_validator(case):
    truth = case['truth']
    def validator(result):
        if not result.submitted or not result.payload: return False
        ans = result.payload.get('answer') or ''
        obj = parse_json_answer(ans if isinstance(ans,str) else json.dumps(ans))
        if not isinstance(obj, dict): return False
        for key, expected in truth.items():
            if key.endswith('_must_contain'):
                actual_key = key[:-len('_must_contain')]
                v = obj.get(actual_key, '')
                if not isinstance(v, str) or expected.lower() not in v.lower():
                    return False
            else:
                v = obj.get(key)
                if isinstance(expected, int):
                    try:
                        if int(v) != expected: return False
                    except Exception: return False
                else:
                    if str(v).strip() != str(expected).strip(): return False
        return True
    return validator
stage('validator_built')


In [ ]:
from fabric_rlm import RLM, FabricLM
from fabric_rlm.experimental import BanditState, EffortBanditPolicy

base_lm = FabricLM('gpt-5', reasoning_effort='minimal', cache=False)
stage('lm_built', base='gpt-5', start_effort='minimal')

CONDITIONS = [('off','0'), ('on','1')]
ablation = summary['ablation']

for case in CASES:
    case_record = {'id': case['id'], 'template': case['template'],
                   'difficulty': case['difficulty'], 'conditions': {}}
    ablation['cases'].append(case_record); write_summary()
    for cond_name, env_val in CONDITIONS:
        os.environ['FABRIC_RLM_PVR'] = env_val
        state = BanditState()
        try:
            validator = make_validator(case)
            policy = EffortBanditPolicy(
                state=state, task_key=case['template'], warmup=2,
                base_lm_spec=base_lm, base_reasoning_effort='minimal',
                parallel_rollouts=3,
            )
            rlm = RLM(
                signature='question -> answer',
                lm=base_lm, engine='adaptive',
                adaptive=dict(policy=policy, validator=validator,
                              max_attempts=6, parallel_rollouts=1),
            )
            t0 = time.perf_counter()
            result = rlm.run({'question': case['question']})
            elapsed = time.perf_counter() - t0
            meta = (result.trajectory.metadata or {}).get('adaptive', {}) if result.trajectory else {}
            attempts = meta.get('attempts', [])
            passed = bool(result.submitted and validator(result))
            cond_record = {
                'passed': passed, 'submitted': result.submitted,
                'elapsed_seconds': elapsed,
                'starting_rung': attempts[0].get('rung') if attempts else None,
                'winner_rung': meta.get('winner_rung'),
                'stop_reason': meta.get('stop_reason'),
                'n_attempts': len(attempts),
                'total_prompt_tokens': sum(a.get('prompt_tokens') or 0 for a in attempts),
                'total_completion_tokens': sum(a.get('completion_tokens') or 0 for a in attempts),
                'final_answer_preview': (str((result.payload or {}).get('answer'))[:400]) if result.payload else None,
            }
        except Exception as exc:
            cond_record = {'passed': False, 'error': repr(exc), 'traceback': traceback.format_exc()}
        case_record['conditions'][cond_name] = cond_record; write_summary()
        stage('cond_done', case=case['id'], cond=cond_name,
              passed=cond_record.get('passed'),
              n_attempts=cond_record.get('n_attempts'),
              elapsed=round(cond_record.get('elapsed_seconds') or 0, 1),
              tokens=(cond_record.get('total_prompt_tokens',0)+cond_record.get('total_completion_tokens',0)))
    stage('case_done', case=case['id'])

ab_table = []
for c in ablation['cases']:
    off = c['conditions'].get('off', {}); on = c['conditions'].get('on', {})
    ab_table.append({'case': c['id'], 'difficulty': c['difficulty'],
        'off_passed': off.get('passed'), 'on_passed': on.get('passed'),
        'off_attempts': off.get('n_attempts'), 'on_attempts': on.get('n_attempts'),
        'off_elapsed': round(off.get('elapsed_seconds') or 0, 1),
        'on_elapsed': round(on.get('elapsed_seconds') or 0, 1),
        'off_tokens': off.get('total_prompt_tokens',0)+off.get('total_completion_tokens',0),
        'on_tokens':  on.get('total_prompt_tokens',0)+on.get('total_completion_tokens',0)})
summary['ab_table'] = ab_table; write_summary()
stage('ablation_done', cases=len(ab_table))
for r in ab_table: print(r)
